In [1]:
# S5 — CKM as CRT entanglement
#
# The CRT maps (a3, a5, a7) → ci on the solenoid circle Z/210Z.
# This mapping ENTANGLES the factors: ci is NOT a simple product of a5 and a7.
# The mass eigenstate depends on ci (through wrapping).
# The gauge eigenstate depends on the wreath product acting on (a3, a5, a7) independently.
# The MISALIGNMENT between ci-ordering and CRT-factor ordering IS the CKM.
#
# Physical picture: the influx flows along the solenoid (ci coordinate).
# The gauge structure acts on the ALGEBRAIC factors (a5, a7) independently.
# But the SPATIAL positions (ci) entangle these factors through CRT.
# A particle with definite gauge quantum numbers (a5, a7) sits at a specific ci.
# The mass depends on ci. So the mass depends on (a5, a7) through CRT — but
# NOT as a simple product. The CRT entanglement creates the mixing.

import numpy as np
from math import prod, gcd

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)

# Build the full CRT map: (a3, a5, a7) → ci
# First need the CRT basis vectors
# ci ≡ a3_residue (mod 3) AND ci ≡ a5_residue (mod 5) AND ci ≡ a7_residue (mod 7)
# where a3_residue = g3^a3, a5_residue = g5^a5, a7_residue = g7^a7
# with generators g3=2, g5=2, g7=3

g3, g5, g7 = 2, 2, 3  # primitive roots

# CRT basis vectors (solutions to ci ≡ 1 (mod p), ci ≡ 0 (mod others))
# e3: e3 ≡ 1 (mod 3), e3 ≡ 0 (mod 5), e3 ≡ 0 (mod 7)  → e3 = 70
# e5: e5 ≡ 0 (mod 3), e5 ≡ 1 (mod 5), e5 ≡ 0 (mod 7)  → e5 = 126
# e7: e7 ≡ 0 (mod 3), e7 ≡ 0 (mod 5), e7 ≡ 1 (mod 7)  → e7 = 120

e3 = 70   # 70 mod 3 = 1, 70 mod 5 = 0, 70 mod 7 = 0
e5 = 126  # 126 mod 3 = 0, 126 mod 5 = 1, 126 mod 7 = 0
e7 = 120  # 120 mod 3 = 0, 120 mod 5 = 0, 120 mod 7 = 1

# Verify
assert e3 % 3 == 1 and e3 % 5 == 0 and e3 % 7 == 0
assert e5 % 3 == 0 and e5 % 5 == 1 and e5 % 7 == 0
assert e7 % 3 == 0 and e7 % 5 == 0 and e7 % 7 == 1

def crt_to_ci(a3, a5, a7):
    """Map CRT indices to ci on the solenoid circle."""
    r3 = pow(g3, a3, 3)  # residue mod 3
    r5 = pow(g5, a5, 5)  # residue mod 5
    r7 = pow(g7, a7, 7)  # residue mod 7
    ci = (r3 * e3 + r5 * e5 + r7 * e7) % P4
    if ci == 0:
        ci = P4  # coprime to 210, so ci=210 wraps to ci=0 which we skip
    return ci

# Build the full ci map for quarks (a3=1)
print('=== CRT map for quarks (a3=1) ===')
print(f'CRT basis vectors: e3={e3}, e5={e5}, e7={e7}')
print(f'\n{"a5":>3} {"a7":>3} {"gen":>4} {"ci":>4}  {"ci/P4":>8}')
print('-' * 30)

gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}
ci_map = {}
for a5 in range(4):
    for a7 in range(6):
        ci = crt_to_ci(1, a5, a7)
        gen = gen_map[a7]
        ci_map[(a5, a7)] = ci
        print(f'{a5:3d} {a7:3d} {gen:4d} {ci:4d}  {ci/P4:8.4f}')

# ══════════════════════════════════════════════════════════════
# THE KEY: How does ci depend on a5 at FIXED generation?
# If the mass depends on ci, and ci changes with a5, then
# the mass eigenstate ROTATES when a5 changes.
# The amount of rotation = the CKM angle.
# ══════════════════════════════════════════════════════════════

print(f'\n=== ci as a function of a5 at fixed generation ===')
print(f'How does the spatial position change under isospin rotation?')

for gen in [1, 2, 3]:
    a7_vals = [a7 for a7 in range(6) if gen_map[a7] == gen]
    print(f'\n  Gen{gen} (a7 ∈ {a7_vals}):')
    for a7_val in a7_vals:
        cis = [ci_map[(a5, a7_val)] for a5 in range(4)]
        print(f'    a7={a7_val}: ci = {cis}  (a5=0,1,2,3)')
        # The ci CHANGE under a5: how much does the position shift?
        for da5 in [1, 2, 3]:
            delta = (ci_map[(da5, a7_val)] - ci_map[(0, a7_val)]) % P4
            if delta > P4//2:
                delta -= P4
            print(f'      a5: 0→{da5}: Δci = {delta}')

# ══════════════════════════════════════════════════════════════
# The isospin step Δci(a5: 0→1) at fixed a7:
# This is the CRT contribution of the a5=1 basis vector.
# Δci = e5 * (g5^1 - g5^0) mod P4 = 126 * (2-1) = 126 for all a7.
# So the isospin step is UNIFORM: Δci = 126 regardless of generation.
# This means the spatial shift under isospin is the SAME for all generations.
# ══════════════════════════════════════════════════════════════

print(f'\n=== Isospin step is uniform: Δci(a5:0→1) = e5 = {e5} ===')
print(f'This is {e5}/{P4} = {e5/P4:.4f} of the solenoid period')
print(f'= {e5/P4:.4f} = p2/p3 = {p2/p3:.4f}? No, 126/210 = 3/5')
print(f'126 = P4 × 3/5 = p1 × p2^2 × p4')

# If the isospin step is uniform, how does the CKM arise?
# The CKM requires GENERATION-DEPENDENT isospin effects.
# Uniform Δci = 126 means ALL generations shift by the same amount.

# BUT: the MASS depends nonlinearly on ci (through wrapping).
# Even though Δci is the same, the mass CHANGE is different because
# different ci values sit in different wrapping regimes.

# ci=11 (gen2 at a5=0) is in the WRAPPING zone
# ci=11+126 = 137 (gen2 at a5=1) is in the NON-WRAPPING zone
# The mass at ci=137 is very different from ci=11!

# ci=41 (gen1 at a5=0, a7=3) is at the wrapping BOUNDARY
# ci=41+126 = 167 (gen1 at a5=1) is in the non-wrapping zone

# ci=191 (gen3 at a5=0, a7=2) is in the non-wrapping zone
# ci=191+126 = 317 mod 210 = 107 (gen3 at a5=1) is also non-wrapping

kappa = 1.0 / np.sqrt(P4)
print(f'\n=== Mass change under isospin step (wrapping nonlinearity) ===')
print(f'The spatial decay at each position:')
for a7_val, gen in [(3, 1), (4, 2), (2, 3)]:
    ci_a5_0 = ci_map[(0, a7_val)]
    ci_a5_1 = ci_map[(1, a7_val)]
    decay_0 = np.exp(-kappa * ci_a5_0)
    decay_1 = np.exp(-kappa * ci_a5_1)
    ratio = decay_0 / decay_1
    print(f'  Gen{gen} a7={a7_val}: ci(a5=0)={ci_a5_0}, ci(a5=1)={ci_a5_1}')
    print(f'    decay(a5=0) = e^(-k*{ci_a5_0}) = {decay_0:.6f}')
    print(f'    decay(a5=1) = e^(-k*{ci_a5_1}) = {decay_1:.6f}')
    print(f'    ratio = {ratio:.4f}')
    # Is this ratio generation-dependent? YES — because ci_a5_0 differs by gen
    # The ratio = e^{-kappa*(ci_a5_0 - ci_a5_1)} = e^{+kappa*126} for all gens
    # Wait — it's e^{-kappa*(ci_a5_0 - ci_a5_1)}. The ci difference is
    # ci_a5_1 - ci_a5_0 = 126 (mod 210). But ci_a5_0 could be > ci_a5_1.
    actual_shift = (ci_a5_1 - ci_a5_0) % P4
    if actual_shift > P4//2:
        actual_shift -= P4
    print(f'    actual shift: {actual_shift} (mod {P4})')

# The wrapping zone is ci < ~43. The non-wrapping zone is ci > ~43.
# Gen2 at a5=0: ci=11 (WRAPPING) → a5=1: ci=137 (non-wrapping)
# Gen1 at a5=0: ci=41 (boundary) → a5=1: ci=167 (non-wrapping)  
# Gen3 at a5=0: ci=191 (non-wrapping) → a5=1: ci=107 (non-wrapping)
#
# Gen2 CROSSES the wrapping boundary under isospin rotation!
# Gen1 barely crosses it. Gen3 stays in the non-wrapping zone.
# This asymmetry is what creates the CKM.

print(f'\n=== The wrapping boundary crossing ===')
ci_boundary = np.log(2) / kappa
print(f'Wrapping boundary: ci ≈ {ci_boundary:.1f}')
print(f'')
for a7_val, gen in [(3, 1), (4, 2), (2, 3)]:
    ci_0 = ci_map[(0, a7_val)]
    ci_1 = ci_map[(1, a7_val)]
    in_wrap_0 = 'WRAPPING' if ci_0 < 44 else 'coherent'
    in_wrap_1 = 'WRAPPING' if ci_1 < 44 else 'coherent'
    crosses = 'CROSSES BOUNDARY' if (ci_0 < 44) != (ci_1 < 44) else 'stays same zone'
    print(f'  Gen{gen}: ci(a5=0)={ci_0:3d} [{in_wrap_0}] → ci(a5=1)={ci_1:3d} [{in_wrap_1}] — {crosses}')

# Gen2 CROSSES from wrapping to coherent. This is the biggest change.
# Gen1 also crosses (ci=41 is borderline wrapping).
# Gen3 stays coherent.
#
# The CKM angle should be determined by HOW MUCH the mass changes
# when gen2 crosses the wrapping boundary under isospin rotation.
# This is a NONLINEAR effect — the wrapping creates a sharp transition.

# The mass at ci=11 (wrapping, large RMS ≈ 1.85) vs ci=137 (coherent, small RMS ≈ 0.24)
# The mass ratio: exp(x_q * ln(RMS_11/RMS_137))... but this is the SAME generation.
# The CKM doesn't compare the mass of gen2 at a5=0 vs a5=1.
# It compares the ORDERING of generations in the two sectors.

# At a5=0: gen2 (ci=11) is HEAVIEST (most wrapping), gen3 (ci=191) is lightest
# At a5=1: gen2 (ci=137) is in the coherent zone — now LIGHTER than gen3 which
#          also moved but stayed coherent (ci=107).

# Wait — let me check the actual RMS at a5=1 crossings from S1 data:
# a5=1 gen2 (ci=17 with a7=1, not ci=137 with a7=4!): RMS = 3.43 (HEAVY)
# a5=1 gen3 (ci=107 with a7=2): RMS = 0.28 (light)
# a5=1 gen1 (ci=167 with a7=3): RMS = 0.25 (light)

# So at a5=1: gen2 is STILL the heaviest (ci=17 has wrapping).
# But gen2 at a5=1 uses a DIFFERENT k7 (a7=1, ci=17) than at a5=0 (a7=4, ci=11)!
# The dominant k7 switched from 4 to 1. This is the Z2 rotation within gen2.

# The CKM doesn't come from the generation ORDERING switching.
# Both a5=0 and a5=1 have gen2 as heaviest.
# The CKM comes from the fact that the mass eigenstate within gen2
# is a DIFFERENT k7 in the two sectors.

# In the DOWN sector: gen2 mass eigenstate = k7=4 (ci=11, wrapping)
# In the UP-1 sector: gen2 mass eigenstate = k7=1 (ci=17, wrapping)
# These are DIFFERENT k7 values within the same generation.
# The rotation from k7=4 to k7=1 within gen2 IS an internal rotation.

# But k7=4 and k7=1 are both gen2. The CKM operates in GENERATION space,
# not k7 space. The internal Z2 rotation doesn't produce inter-generation mixing.

# UNLESS: the k7 rotation LEAKS into the generation space through
# the wrapping nonlinearity. At the wrapping boundary, the mass
# eigenstate is a SUPERPOSITION of multiple k7 values from DIFFERENT
# generations (because the wrapping mixes all 7 sheets of the p=7 covering).

print(f'\n=== The wrapping as generation mixer ===')
print(f'At ci=11 (wrapping zone), 85.7% of branches wrap.')
print(f'The wrapped branches lose their k7 identity — they become')
print(f'uniformly distributed on [-pi, pi] regardless of initial j3.')
print(f'Only the 14.3% unwrapped branches (j3=0) retain coherence.')
print(f'')
print(f'The wrapping MIXES j3 values → mixes generation identity.')
print(f'A particle at ci=11 is not purely gen2 — it is 1/7 gen2')
print(f'(from the j3=0 sheet) plus 6/7 generation-mixed (from wrapped sheets).')
print(f'')
print(f'At ci=137 (coherent zone), 0% wrap. All j3 sheets retain identity.')
print(f'A particle at ci=137 is purely gen2.')
print(f'')
print(f'The CKM angle = the generation PURITY DIFFERENCE between the')
print(f'wrapping zone (a5=0) and the coherent zone (a5=1 at high ci).')
print(f'')
print(f'At a5=0 gen2 (ci=11): generation purity = 1/7 (only j3=0 is coherent)')
print(f'At a5=1 gen2 (ci=17): generation purity = 4/15 (wrapping at R3 is 73.3%)')
print(f'')
print(f'The mixing angle = the change in generation purity:')
nwf_11 = 1/7  # non-wrapping fraction at R3 for ci=11
nwf_17 = 4/15  # non-wrapping fraction at R3 for ci=17
print(f'  nwf(ci=11) = 1/7 = {nwf_11:.6f}')
print(f'  nwf(ci=17) = 4/15 = {nwf_17:.6f}')
print(f'  Difference: {nwf_17 - nwf_11:.6f}')
print(f'  Ratio: {nwf_17/nwf_11:.6f} = (4/15)/(1/7) = 28/15')
print(f'')
print(f'  sin(theta_C) from purity change:')
print(f'    nwf_17 - nwf_11 = {nwf_17-nwf_11:.6f} (cf 0.225)')
print(f'    sqrt(nwf_17 - nwf_11) = {np.sqrt(nwf_17-nwf_11):.6f}')
print(f'    sqrt(nwf_11 * nwf_17) = {np.sqrt(nwf_11*nwf_17):.6f}')
print(f'    sqrt(nwf_11 / nwf_17) = {np.sqrt(nwf_11/nwf_17):.6f}')

# sqrt(nwf_11 * nwf_17) = sqrt((1/7)(4/15)) = sqrt(4/105) = 2/sqrt(105) = 0.1952
# PDG: 0.22500. Off by 13%.

# (nwf_17 - nwf_11) = 4/15 - 1/7 = (28-15)/105 = 13/105 = 0.1238
# sqrt(13/105) = 0.352. Off by 56%.

# Hmm. None of these directly give the Cabibbo angle.

# But the PHYSICAL picture is now clear:
# The CKM arises because the wrapping MIXES generations.
# At the wrapping zone (ci < 44), the branches lose their j3 identity.
# The mass eigenstate at a wrapping crossing is NOT a pure generation state.
# It's a SUPERPOSITION weighted by the non-wrapping fraction.
# The mixing angle between two crossings (one wrapping, one less wrapping)
# is determined by the DIFFERENCE in generation purity.

print(f'\n=== THE CKM MECHANISM ===')
print(f'1. The solenoid has 11 wrapping crossings where branches lose j3 identity')
print(f'2. A particle at a wrapping crossing is NOT purely one generation')
print(f'   — it is (non-wrapping fraction) pure + (wrapping fraction) mixed')
print(f'3. The DOWN mass eigenstate (ci=11, 85.7% wrapped) is only 14.3% pure gen2')
print(f'4. The UP mass eigenstate (ci=17, 73.3% wrapped) is 26.7% pure gen2')
print(f'5. The DIFFERENCE in purity between DOWN and UP = the CKM mixing')
print(f'6. The mixing angle depends on the wrapping fractions at the two crossings')
print(f'7. These fractions come from the spatial dynamics (resonance + comb structure)')
print(f'')
print(f'This is the same mechanism as mass — mass = total coherence across levels,')
print(f'mixing = DIFFERENTIAL coherence between sectors at different spatial positions.')

=== CRT map for quarks (a3=1) ===
CRT basis vectors: e3=70, e5=126, e7=120

 a5  a7  gen   ci     ci/P4
------------------------------
  0   0    1  176    0.8381
  0   1    2  206    0.9810
  0   2    3   86    0.4095
  0   3    1  146    0.6952
  0   4    2  116    0.5524
  0   5    3   26    0.1238
  1   0    1   92    0.4381
  1   1    2  122    0.5810
  1   2    3    2    0.0095
  1   3    1   62    0.2952
  1   4    2   32    0.1524
  1   5    3  152    0.7238
  2   0    1  134    0.6381
  2   1    2  164    0.7810
  2   2    3   44    0.2095
  2   3    1  104    0.4952
  2   4    2   74    0.3524
  2   5    3  194    0.9238
  3   0    1    8    0.0381
  3   1    2   38    0.1810
  3   2    3  128    0.6095
  3   3    1  188    0.8952
  3   4    2  158    0.7524
  3   5    3   68    0.3238

=== ci as a function of a5 at fixed generation ===
How does the spatial position change under isospin rotation?

  Gen1 (a7 ∈ [0, 3]):
    a7=0: ci = [176, 92, 134, 8]  (a5=0,1,2,3)
      a5: 

In [1]:
# S4 — The F-N correction: does the a5 perturbation modify m_s/m_d?
#
# The cascade gives m_s/m_d = 20.000 at a5=0.
# F-N gives sin(theta_C) = 1/sqrt(m_s/m_d) = 1/sqrt(20) = 0.2236.
# PDG: sin(theta_C) = 9/40 = 0.2250. Requires m_s/m_d = (40/9)^2 = 19.753.
#
# The PHYSICAL m_s/m_d might not be the a5=0 CP ratio alone.
# The weak interaction mixes a5 sectors, so the physical mass is a
# COMBINATION of the a5=0 mass eigenvalue with a5=1,3 corrections.
#
# The correction: delta(m_s/m_d) = rho * (a5-dependent contribution)
# If this shifts m_s/m_d from 20.0 to 19.753, the F-N route works exactly.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
P3 = p1 * p2 * p3
kappa = 1.0 / np.sqrt(P4)
rho = kappa
phi_P3 = (p1-1)*(p2-1)*(p3-1)

print('=== The F-N correction problem ===')
print(f'Cascade m_s/m_d (a5=0) = 20.000')
print(f'F-N target: m_s/m_d = (40/9)^2 = {(40/9)**2:.6f} = (phi(P3)*p3/p2^2)^2')
print(f'Difference: {20 - (40/9)**2:.6f} = {(20 - (40/9)**2)/20*100:.3f}%')
print(f'Needed correction: {((40/9)**2 - 20)/20*100:.3f}% = -{(20-(40/9)**2)/(40/9)**2*100:.3f}%')

# The a5=0 mass ratio: CP_R3(a5=0)^x_q = 6.607^1.587 = 20.00
# The a5=1 CP ratio: 0.897 (< 1, inverted!)
# The a5=3 CP ratio: 0.428 (< 1, inverted!)
# The a5=2 CP ratio: 1.008 (≈ 1, protected)

# Physical mass includes the a5 correction from the weak interaction.
# In the SM, the pole mass = bare mass + loop corrections.
# In the solenoid, the physical mass = a5=0 mass × (1 + rho * correction)

# The correction from a5=1: the isospin-1 sector's CP ratio acts as
# a multiplicative correction to the mass ratio.
# From S2 data: the a5-dependent contribution to gen2 S_eff at k3=1:
#   k7=1: a5_contrib = -0.634 (from Im_2 - Im_1)
#   k7=4: a5_contrib = -1.366

# The effective mass correction from the a5 perturbation:
# dm/m = rho * (a5_contrib) / S_eff(a5=0)
# For the dominant mass eigenstate (k7=4 at a5=0):
# dm/m(gen2) = rho * (-1.366) / 0.866 = -0.069 * 1.577 = -0.109

# For gen3 (k7=5 at a5=0):
# a5_contrib(k7=5) at a5=1: -2.366
# dm/m(gen3) = rho * (-2.366) / 0.866 = -0.069 * 2.732 = -0.188

# The RATIO correction:
# d(m_s/m_d)/d(rho) = m_s/m_d × [dm_s/m_s - dm_d/m_d]
# But gen1 (d quark) has S_eff = 0 at a5=0, so dm_d involves the a5 correction
# lifting gen1 from zero.

print(f'\n=== a5 correction to mass ratio ===')
# From S2 data for k3=1 (right-chirality quarks):
# gen1 at a5=0: S_eff = 0 for both k7=0 and k7=3
# gen1 at a5=1: S_eff = +0.069 (k7=0), +0.207 (k7=3)
# The a5 perturbation LIFTS gen1 from zero by rho * a5_contrib

# Gen1 a5_contrib: k7=0 gets +1.000, k7=3 gets +3.000 (from S2)
# Gen2 a5_contrib: k7=1 gets -0.634, k7=4 gets -1.366
# Gen3 a5_contrib: k7=2 gets +0.366, k7=5 gets -2.366

# The dominant eigenstate of gen1 at a5=0 is k7=3 (ci=41, RMS=2.59).
# Its a5 correction at k5=1: S_eff shifts from 0 to +0.207 = rho * 3.000
# So the gen1 mass gets a correction: m_d(phys) = m_d(bare) * v^{rho * 3}

# The dominant eigenstate of gen2 at a5=0 is k7=4 (ci=11, RMS=3.65).
# Its a5 correction at k5=1: S_eff shifts from +0.866 to +0.772
# Change: -0.094 = rho * (-1.366)

# In the tower product mass formula: m ~ v^{|S_eff|}
# The mass RATIO m_s/m_d changes because both S_eff values change:
# m_s/m_d(phys) = v^{|S_eff(gen2)|} / v^{|S_eff(gen1)|}
# = v^{0.866 + rho*(-1.366)} / v^{0 + rho*(3.000)}   [approximate]
# = v^{0.866 - 0.094} / v^{0.207}
# = v^{0.772 - 0.207}
# = v^{0.565}

# But at a5=0: m_s/m_d = v^{0.866 - 0} = v^{0.866}
# So the RATIO of mass ratios: (physical)/(bare) = v^{0.565} / v^{0.866} = v^{-0.301}

# This is the WRONG direction — the a5 correction makes the mass ratio SMALLER.
# v < 1 for exponential mass, so v^{-0.301} > 1, meaning m_s/m_d INCREASES.
# But we need it to DECREASE from 20.0 to 19.75.

# Wait — this analysis assumes the UP and DOWN sectors contribute ADDITIVELY
# to the physical mass. That might not be right.

# Let me think about this more carefully.
# In the SM, the CKM comes from the MISALIGNMENT between up and down
# Yukawa matrices. The masses themselves are the eigenvalues of M†M.
# The Cabibbo angle comes from sin θ_C = √(m_d/m_s) for SPECIFIC
# mass matrix textures (nearest-neighbor, etc.).

# In the solenoid:
# - The a5=0 sector gives the BARE mass hierarchy: m_s/m_d = 20
# - The a5=1,3 sectors provide the WEAK COUPLING between generations
# - The Cabibbo angle relates to how the weak coupling ROTATES the mass basis

# The F-N relation sin θ_C = √(m_d/m_s) is not about correcting the mass ratio.
# It's about the TEXTURE of the mass matrix. Specifically:
# M_d = [ 0   a ]    gives eigenvalues ∝ {a², b²} and sin θ_C = a/b
#        [ a   b ]
# where b >> a, so m_d ∝ a²/b and m_s ∝ b, giving m_s/m_d ∝ b²/a² and sin θ_C = a/b = √(m_d/m_s).

# In the solenoid, this texture comes from the directed splitting:
# - The diagonal elements b = S_eff(gen2) and 0 = S_eff(gen1) at a5=0
# - The off-diagonal element a comes from the a5 perturbation connecting gen1 and gen2
# - a ∝ rho × (a5-dependent coupling between gen1 and gen2)

# From S2: the a5 correction to gen1 at k5=1 is rho × 3.000 (for k7=3)
# The a5 correction to gen2 at k5=1 is rho × (-1.366) (for k7=4)
# The OFF-DIAGONAL coupling between gen1 and gen2 would involve these corrections

# In the F-N texture: sin θ_C = a/b where
# b = S_eff(gen2, a5=0) = 0.866 = √3/2
# a = ??? (the off-diagonal element from the a5 perturbation)

# If sin θ_C = a/b = 0.225, then a = 0.225 × 0.866 = 0.195
# Is a = rho × something?
# rho × √3 = 0.069 × 1.732 = 0.120 (the gen2-gen3 splitting)
# rho × 3 = 0.207 (the gen1 lifting at k7=3)

# Hmm, a = 0.195 is between rho × √3 (0.120) and rho × 3 (0.207).
# Average: (0.120 + 0.207)/2 = 0.163. Not 0.195.
# Geometric mean: sqrt(0.120 × 0.207) = sqrt(0.0248) = 0.158. Not 0.195.

# Let me try: a = rho × gen1_a5_contrib(dominant) × gen2_mass_factor
# a = rho × 3.000 × ??? 

# Actually, the off-diagonal matrix element between gen1 and gen2
# should involve the OVERLAP of gen1 and gen2 states under the a5 perturbation.
# This overlap depends on the specific k7 values and their CRT structure.

# From the CRT: gen1 k7=3 (ci=41) and gen2 k7=4 (ci=11) at a5=0.
# At a5=1: gen1 k7=3 goes to ci=? and gen2 k7=4 goes to ci=?.
# The a5 change maps: (a3=1, a5=0, a7=3) → (a3=1, a5=1, a7=3) 
# ci changes from 41 to 167 (from S1 data: a5=1, a7=3, ci=167)
# And gen2: (a3=1, a5=0, a7=4) → (a3=1, a5=1, a7=4), ci=137

# The OVERLAP between the a5=0 and a5=1 states at these k7 values:
# This is the spatial overlap between ci=41 (a5=0) and ci=167 (a5=1) profiles
# and between ci=11 (a5=0) and ci=137 (a5=1) profiles.

# But we showed earlier that these spatial overlaps are all near 1.
# So the a5 perturbation doesn't create much OFF-DIAGONAL mixing in generation space.

# CONCLUSION: The F-N texture requires an OFF-DIAGONAL mass matrix element 'a'
# between gen1 and gen2. In the solenoid, the mass matrix IS diagonal in the 
# CRT basis (each character has a definite mass). There are NO off-diagonal elements.
# The F-N relation sin θ_C = √(m_d/m_s) applies only to SPECIFIC textures.
# The solenoid has a DIAGONAL mass matrix, not a nearest-neighbor texture.

print(f'\n=== Conclusion: F-N texture and the diagonal mass matrix ===')
print(f'The F-N relation sin(theta_C) = sqrt(m_d/m_s) requires a specific')
print(f'mass matrix texture with off-diagonal elements. The solenoid mass')
print(f'matrix is DIAGONAL in the CRT basis — each (a3, a5, a7) character')
print(f'has a definite mass with no mixing.')
print(f'')
print(f'The CKM must arise from the WEAK INTERACTION, not the mass matrix.')
print(f'The weak boson W couples different a5 sectors (0↔1, 0↔3).')
print(f'The coupling matrix elements between a5 sectors, projected onto')
print(f'the mass eigenbasis, give the CKM.')
print(f'')
print(f'The directed Cayley perturbation (NB59) provides exactly this:')
print(f'it is the non-real operator that couples between sectors.')
print(f'The CKM elements are the DIRECTED SPLITTING DIFFERENCES between')
print(f'the a5=0 and a5=1 sectors, not the spatial overlaps.')

# ══════════════════════════════════════════════════════════════
# THE PROPER CKM COMPUTATION:
# The W boson couples gen_i(down, a5=0) to gen_j(up, a5=1).
# The coupling strength = the directed Cayley matrix element
# connecting these two characters.
# CKM_{ij} = <gen_i, a5=0 | A_g | gen_j, a5=1> / normalization
# where A_g is the directed operator.
# ══════════════════════════════════════════════════════════════

print(f'\n=== The W boson as directed Cayley coupling ===')
print(f'The W couples a5=0 (down) to a5=1 (up) via the directed operator.')
print(f'A_g = B_g - B_{{g^{{-1}}}} for generators g in {{17, 23, 37}}.')
print(f'')
print(f'In the character basis, A_g is diagonal:')
print(f'  <chi_k | A_g | chi_k> = 2i * Im(chi_k(g))')
print(f'')
print(f'But the W CHANGES a5: it connects (a3, a5=0, a7) to (a3, a5=1, a7*).')
print(f'The a5 change means the two characters have DIFFERENT k5 labels.')
print(f'A_g is diagonal in the FULL character basis — so the coupling is ZERO')
print(f'between characters with different k5!')
print(f'')
print(f'Wait — this means A_g CANNOT couple different a5 sectors.')
print(f'The directed operator acts WITHIN each a5 sector, not between them.')
print(f'The CKM cannot come from the directed Cayley operator directly.')

# This is a fundamental issue. The directed operator is diagonal in the
# character basis and cannot couple different a5 values. The W boson
# in the SM couples different isospin components — but in the solenoid,
# the a5 (isospin) sectors are ORTHOGONAL in the character basis.

# The CKM must arise from a DIFFERENT mechanism. In the SM, the Yukawa
# coupling (not the gauge coupling) creates the mixing. The gauge interaction
# is flavor-diagonal — it's the MASS MATRIX that creates the misalignment.

# In the solenoid: the gauge structure (wreath product, NB140-144) determines
# which characters couple to which gauge bosons. The mass structure (wrapping,
# NB161-162) determines the mass eigenstates. If the gauge eigenstates and
# mass eigenstates are different bases, there's mixing.

# The GAUGE eigenstates at the p=5 level are the a5 values:
# a5=0 and a5=2 form one doublet (down-type + protected)
# a5=1 and a5=3 form another doublet (isospin pair)

# The MASS eigenstates are determined by the wrapping at a5=0 crossings.
# The mass eigenstates DON'T depend on a5 — they depend on a7 (generation).

# The CKM = the rotation between the GAUGE basis (a5 defines up/down)
# and the MASS basis (a7 defines generation) within the SAME particle.

# Actually, for a quark, the full quantum numbers are (a3, a5, a7).
# The mass depends primarily on a7 (generation) through the wrapping.
# The weak interaction depends primarily on a5 (isospin) through the gauge.
# The CKM mixes a7 and a5 — it's the cross-talk between generation and isospin.

# This cross-talk comes from the CRT structure itself: a5 and a7 are
# INDEPENDENT quantum numbers in Z*_210 = Z1 × Z2 × Z4 × Z6.
# But the ci values (which determine masses) depend on ALL quantum numbers
# through the CRT reconstruction. So the mass DOES depend on a5 through ci.

# The CKM angle = the amount by which the ci value (and hence the mass)
# changes when a5 changes, RELATIVE to the change when a7 changes.

print(f'\n=== CKM from CRT cross-talk between a5 and a7 ===')

# For quark (a3=1), the ci values at different (a5, a7):
# The ci is the CRT reconstruction of (a3=1, a5, a7) into Z*_210.
# When we change a5 from 0 to 1 at fixed a7, ci changes.
# When we change a7 from gen2 to gen1 at fixed a5, ci changes.
# The RATIO of these changes determines the mixing angle.

# ci values for quark gen1 and gen2 at a5=0 and a5=1:
ci_d1 = 41   # (a3=1, a5=0, a7=3) = gen1 dominant at a5=0
ci_d2 = 11   # (a3=1, a5=0, a7=4) = gen2 dominant at a5=0
ci_u1 = 167  # (a3=1, a5=1, a7=3) = gen1 at a5=1
ci_u2 = 17   # (a3=1, a5=1, a7=1) = gen2 dominant at a5=1 (NOTE: different a7!)

print(f'ci values:')
print(f'  DOWN gen1 (a5=0, a7=3): ci={ci_d1}')
print(f'  DOWN gen2 (a5=0, a7=4): ci={ci_d2}')
print(f'  UP   gen1 (a5=1, a7=3): ci={ci_u1}')
print(f'  UP   gen2 (a5=1, a7=1): ci={ci_u2}')

# The "generation step" in ci-space: ci(gen2) - ci(gen1)
gen_step_down = ci_d2 - ci_d1  # 11 - 41 = -30 = -P3
gen_step_up = ci_u2 - ci_u1    # 17 - 167 = -150

# The "isospin step" in ci-space: ci(a5=1) - ci(a5=0) at fixed (a3, a7)
iso_step_gen1 = ci_u1 - ci_d1  # 167 - 41 = 126
iso_step_gen2_same_a7 = 137 - 11  # ci(a5=1,a7=4) - ci(a5=0,a7=4) = 126

print(f'\nGeneration step (ci change for gen2→gen1):')
print(f'  DOWN: {gen_step_down} = -P3 = -{P3}')
print(f'  UP:   {gen_step_up}')
print(f'\nIsospin step (ci change for a5=0→a5=1 at fixed a7):')
print(f'  gen1 (a7=3): {iso_step_gen1}')
print(f'  gen2 (a7=4): {iso_step_gen2_same_a7}')

# The isospin step is 126 for both generations (at fixed a7).
# 126 = P4 * 3/5 = 210 * 3/5 = 126. Or 126 = P3 * 4.2 = not clean.
# 126 = 2 * 63 = 2 * 9 * 7 = p1 * p2^2 * p4
print(f'  126 = p1 * p2^2 * p4 = {p1 * p2**2 * p4}')

# The KEY: the generation step and isospin step in ci-space encode
# the CRT structure. The ratio of these steps determines how much
# the mass changes under isospin rotation vs generation rotation.

# The SPATIAL DECAY between crossings:
# Mass depends on e^{-kappa*ci}. The mass RATIO for a generation step:
# m(gen2)/m(gen1) ~ e^{-kappa*P3} at fixed a5

# And the mass CHANGE under isospin step:
# m(a5=1)/m(a5=0) ~ e^{-kappa*126} at fixed a7

decay_gen = np.exp(-kappa * abs(gen_step_down))  # P3 = 30 step
decay_iso = np.exp(-kappa * iso_step_gen1)        # 126 step

print(f'\nSpatial decay factors:')
print(f'  Generation step (P3=30): e^(-kappa*30) = {decay_gen:.6f}')
print(f'  Isospin step (126): e^(-kappa*126) = {decay_iso:.6f}')
print(f'  Ratio (iso/gen): {decay_iso/decay_gen:.6f}')

# The Cabibbo angle might be related to this ratio:
# sin(theta_C) ~ decay_iso / decay_gen
ratio = decay_iso / decay_gen
print(f'\n  decay_iso / decay_gen = {ratio:.6f}')
print(f'  sqrt(ratio) = {np.sqrt(ratio):.6f}')
print(f'  PDG sin(theta_C) = 0.22500')

# Also try: the ratio of the isospin step to the period
print(f'\n  Isospin step / P4 = 126/210 = {126/210:.6f} = 3/5 = p2/p3')
print(f'  Generation step / P4 = 30/210 = {30/210:.6f} = 1/7 = 1/p4')

# sin(theta_C) as a ratio of CRT steps:
# 126/210 = 3/5 and 30/210 = 1/7
# (1/7)/(3/5) = 5/21 = 0.238. Close to 0.225?
print(f'  (1/p4)/(p2/p3) = p3/(p2*p4) = 5/21 = {5/21:.6f}')
print(f'  vs sin(theta_C) = 0.22500. Deviation: {abs(5/21 - 0.225)/0.225*100:.1f}%')

# 5/21 = 0.238 is 5.8% from 0.225. Not bad but not exact.

# What about: sin(theta_C) = gen_step / (gen_step + iso_step)?
# = 30 / (30 + 126) = 30/156 = 5/26 = 0.1923. Off by 14%.

# Or: sin^2(theta_C) = gen_step / iso_step = 30/126 = 5/21 = 0.238
# Wait, I already checked that. And sin^2(0.225) = 0.0506.
# 30/126 = 0.238 ≠ 0.0506.

# The F-N route is still the best: sin θ_C = 1/√(m_s/m_d) = 1/√20 = 0.2236
# with a 2.1σ deviation. The CRT cross-talk gives 5/21 = 0.238 (5.8% off).

print(f'\n=== Summary of CKM approaches ===')
print(f'1. Froggatt-Nielsen: sin(theta_C) = 1/sqrt(20) = 0.2236 (2.1σ from PDG)')
print(f'2. CRT step ratio: p3/(p2*p4) = 5/21 = 0.2381 (5.8% from PDG)')
print(f'3. Directed splitting: gives gen2-gen3 asymmetry, not gen1-gen2 Cabibbo')
print(f'4. Spatial overlap: all near 1, not discriminating')
print(f'5. Pattern matching nwf ratios: finds matches but no mechanism')
print(f'')
print(f'BEST RESULT: F-N from cascade masses, 2.1σ off.')
print(f'')
print(f'The CKM mechanism in the solenoid:')
print(f'  - Masses determined by wrapping at a5=0 (diagonal mass matrix)')
print(f'  - Weak interaction couples a5 sectors through gauge structure')
print(f'  - The coupling is flavor-diagonal in the CRT basis')
print(f'  - The CKM arises from the TEXTURE of the effective mass matrix')
print(f'    after integrating out the weak coupling (F-N mechanism)')
print(f'  - The cascade gives m_s/m_d = 20 → sin theta_C = 0.2236')
print(f'  - The exact value 9/40 requires either:')
print(f'    (a) a correction to m_s/m_d from -1.25%')  
print(f'    (b) a correction to the F-N relation itself')
print(f'    (c) a deeper mechanism beyond nearest-neighbor texture')

=== The F-N correction problem ===
Cascade m_s/m_d (a5=0) = 20.000
F-N target: m_s/m_d = (40/9)^2 = 19.753086 = (phi(P3)*p3/p2^2)^2
Difference: 0.246914 = 1.235%
Needed correction: -1.235% = -1.250%

=== a5 correction to mass ratio ===

=== Conclusion: F-N texture and the diagonal mass matrix ===
The F-N relation sin(theta_C) = sqrt(m_d/m_s) requires a specific
mass matrix texture with off-diagonal elements. The solenoid mass
matrix is DIAGONAL in the CRT basis — each (a3, a5, a7) character
has a definite mass with no mixing.

The CKM must arise from the WEAK INTERACTION, not the mass matrix.
The weak boson W couples different a5 sectors (0↔1, 0↔3).
The coupling matrix elements between a5 sectors, projected onto
the mass eigenbasis, give the CKM.

The directed Cayley perturbation (NB59) provides exactly this:
it is the non-real operator that couples between sectors.
The CKM elements are the DIRECTED SPLITTING DIFFERENCES between
the a5=0 and a5=1 sectors, not the spatial overlaps.

===

In [1]:
# S3 — CKM from mass eigenstate overlap between a5 sectors
#
# The mass eigenstate for each generation is determined by the SPATIAL
# PROFILE at that generation's crossing in the relevant a5 sector.
# 
# DOWN-type quarks: mass eigenstates from a5=0 spatial profiles
# UP-type quarks: mass eigenstates from a5=1 (or a5=3) spatial profiles
#
# The CKM matrix V_{ij} = <down_i | up_j> where the inner product
# is over the 4D spatial amplitude space (R0, R1, R2, R3).
#
# But the key subtlety: within each generation, there are TWO k7 values
# (Z2 cosets). The mass eigenstate is the HEAVIER one (larger RMS).
# So the "down gen2" eigenstate is the k7 value in the a5=0 sector
# that has larger RMS, and "up gen2" is the k7 value in the a5=1 sector
# that has larger RMS. If these are DIFFERENT k7 values, there's mixing.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
P3 = p1 * p2 * p3
kappa = 1.0 / np.sqrt(P4)
rho = kappa

sys0 = SolenoidSystem(primes=primes)
branches = sys0.all_branches()
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()
res = sys0.integrate_all_branches(branches, cis.astype(float), float(P4+1), backend='jax')

# Compute RMS at all crossings
rms = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in branches])
        Rk_w = np.mod(Rk, 2*np.pi)
        Rk_w[Rk_w > np.pi] -= 2*np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}

def get_idx(a3v, a5v, a7v):
    mask = (a3 == a3v) & (a5 == a5v) & (a7 == a7v)
    where = np.where(mask)[0]
    return where[0] if len(where) > 0 else None

# ══════════════════════════════════════════════════════════════
# For each a5 sector: identify the mass eigenstate per generation
# The mass eigenstate is the k7 VALUE that has larger RMS (heavier)
# within each generation's Z2 pair.
# ══════════════════════════════════════════════════════════════

print('=== Mass eigenstates by sector ===')
print('For each (a5, gen), the Z2 pair has two k7 values.')
print('The mass eigenstate is the one with larger |RMS| (4D norm).')

# k7 pairs per generation: gen1={0,3}, gen2={1,4}, gen3={2,5}
k7_pairs = {1: (0, 3), 2: (1, 4), 3: (2, 5)}

eigenstates = {}  # eigenstates[a5][gen] = (dominant_k7, 4D_vector, subdominant_k7, 4D_vector)

for a5_val in [0, 1, 3]:
    eigenstates[a5_val] = {}
    label = {0: 'DOWN', 1: 'UP-1', 3: 'UP-3'}[a5_val]
    print(f'\n  a5={a5_val} ({label}):')
    
    for gen in [1, 2, 3]:
        k7_a, k7_b = k7_pairs[gen]
        idx_a = get_idx(1, a5_val, k7_a)  # quark (a3=1)
        idx_b = get_idx(1, a5_val, k7_b)
        
        if idx_a is None or idx_b is None:
            continue
        
        r_a = rms[idx_a]
        r_b = rms[idx_b]
        norm_a = np.linalg.norm(r_a)
        norm_b = np.linalg.norm(r_b)
        
        if norm_a >= norm_b:
            dom_k7, dom_vec = k7_a, r_a
            sub_k7, sub_vec = k7_b, r_b
        else:
            dom_k7, dom_vec = k7_b, r_b
            sub_k7, sub_vec = k7_a, r_a
        
        eigenstates[a5_val][gen] = {
            'dom_k7': dom_k7, 'dom_vec': dom_vec, 'dom_norm': np.linalg.norm(dom_vec),
            'sub_k7': sub_k7, 'sub_vec': sub_vec, 'sub_norm': np.linalg.norm(sub_vec),
        }
        
        print(f'    gen{gen}: dominant k7={dom_k7} (|v|={np.linalg.norm(dom_vec):.4f}), '
              f'subdominant k7={sub_k7} (|v|={np.linalg.norm(sub_vec):.4f})')

# ══════════════════════════════════════════════════════════════
# Check: do the dominant k7 values DIFFER between DOWN and UP?
# If yes → the mass eigenstates are misaligned → CKM mixing
# ══════════════════════════════════════════════════════════════

print(f'\n=== Eigenstate misalignment (dominant k7 comparison) ===')
for gen in [1, 2, 3]:
    d = eigenstates[0][gen]['dom_k7']
    u1 = eigenstates[1][gen]['dom_k7']
    u3 = eigenstates[3][gen]['dom_k7']
    match_1 = 'SAME' if d == u1 else 'DIFFERENT → MIXING'
    match_3 = 'SAME' if d == u3 else 'DIFFERENT → MIXING'
    print(f'  gen{gen}: DOWN k7={d}, UP-1 k7={u1} ({match_1}), UP-3 k7={u3} ({match_3})')

# ══════════════════════════════════════════════════════════════
# CKM = overlap matrix between DOWN and UP mass eigenstates
# V_{ij} = <down_i | up_j> where vectors are normalized 4D amplitudes
# ══════════════════════════════════════════════════════════════

def normalize(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-10 else v

print(f'\n=== CKM overlap matrix ===')

# Use the DOMINANT eigenstates for each generation
for up_sector, up_label in [(1, 'UP-1'), (3, 'UP-3')]:
    print(f'\n  V = <DOWN_i | {up_label}_j> (using dominant k7):')
    V = np.zeros((3, 3))
    for di, gen_d in enumerate([1, 2, 3]):
        d_vec = normalize(eigenstates[0][gen_d]['dom_vec'])
        for uj, gen_u in enumerate([1, 2, 3]):
            u_vec = normalize(eigenstates[up_sector][gen_u]['dom_vec'])
            V[di, uj] = np.dot(d_vec, u_vec)
    
    print(f'        {"gen1_up":>10} {"gen2_up":>10} {"gen3_up":>10}')
    for di, gen_d in enumerate([1, 2, 3]):
        print(f'  gen{gen_d}_d {V[di,0]:10.6f} {V[di,1]:10.6f} {V[di,2]:10.6f}')
    
    # Extract mixing angles
    # sin(theta_12) ≈ |V[0,1]| or the angle between gen1_d and gen2_u
    # But this matrix isn't unitary — need to think about what to extract
    
    # The MISALIGNMENT angle between same-generation eigenstates:
    print(f'\n  Same-generation overlaps (should be ~1 if no mixing):')
    for gen in [1, 2, 3]:
        d_vec = normalize(eigenstates[0][gen]['dom_vec'])
        u_vec = normalize(eigenstates[up_sector][gen]['dom_vec'])
        overlap = np.dot(d_vec, u_vec)
        angle = np.arccos(np.clip(abs(overlap), 0, 1))
        print(f'    gen{gen}: <d|u> = {overlap:.6f}, angle = {np.degrees(angle):.4f}°, sin = {np.sin(angle):.6f}')

# ══════════════════════════════════════════════════════════════
# The CKM also involves the SUBDOMINANT eigenstates.
# Within each generation, the mass eigenstate is a SUPERPOSITION
# of the dominant and subdominant k7 values, weighted by their RMS.
# The mixing angle WITHIN a generation determines how much the
# subdominant k7 contributes.
# ══════════════════════════════════════════════════════════════

print(f'\n=== Intra-generation mixing (dominant/subdominant weight) ===')
for a5_val, label in [(0, 'DOWN'), (1, 'UP-1'), (3, 'UP-3')]:
    print(f'\n  {label}:')
    for gen in [1, 2, 3]:
        e = eigenstates[a5_val][gen]
        total = e['dom_norm'] + e['sub_norm']
        dom_weight = e['dom_norm'] / total
        sub_weight = e['sub_norm'] / total
        # The "intra-gen angle" = arctan(sub_weight/dom_weight)
        intra_angle = np.arctan2(sub_weight, dom_weight)
        print(f'    gen{gen}: dom_weight={dom_weight:.4f}, sub_weight={sub_weight:.4f}, '
              f'intra_angle={np.degrees(intra_angle):.2f}°')

# ══════════════════════════════════════════════════════════════
# Construct PROPER mass eigenstates as weighted superpositions
# ══════════════════════════════════════════════════════════════

print(f'\n=== Proper mass eigenstates (weighted by RMS norm) ===')

# For each sector, the mass eigenstate of gen_i is:
# |gen_i> = (norm_dom * dom_hat + norm_sub * sub_hat) / total_norm
# where dom_hat and sub_hat are normalized 4D vectors

proper_states = {}
for a5_val in [0, 1, 3]:
    proper_states[a5_val] = {}
    for gen in [1, 2, 3]:
        e = eigenstates[a5_val][gen]
        # The mass eigenstate is the HEAVIER superposition
        # Weight by norm (larger RMS = more mass contribution)
        state = e['dom_norm'] * normalize(e['dom_vec']) + e['sub_norm'] * normalize(e['sub_vec'])
        proper_states[a5_val][gen] = normalize(state)

# Overlap matrix with proper states
print(f'\n=== CKM from proper weighted eigenstates ===')
for up_sector, up_label in [(1, 'UP-1'), (3, 'UP-3')]:
    print(f'\n  V = <DOWN_i | {up_label}_j>:')
    V = np.zeros((3, 3))
    for di, gen_d in enumerate([1, 2, 3]):
        d_vec = proper_states[0][gen_d]
        for uj, gen_u in enumerate([1, 2, 3]):
            u_vec = proper_states[up_sector][gen_u]
            V[di, uj] = np.dot(d_vec, u_vec)
    
    print(f'        {"gen1_up":>10} {"gen2_up":>10} {"gen3_up":>10}')
    for di, gen_d in enumerate([1, 2, 3]):
        print(f'  gen{gen_d}_d {V[di,0]:10.6f} {V[di,1]:10.6f} {V[di,2]:10.6f}')
    
    # Same-generation overlaps
    print(f'  Diagonal overlaps: [{V[0,0]:.6f}, {V[1,1]:.6f}, {V[2,2]:.6f}]')
    
    # Off-diagonal = mixing
    print(f'  Off-diagonal (mixing):')
    for di in range(3):
        for uj in range(3):
            if di != uj:
                angle = np.arccos(np.clip(abs(V[di,di]) * abs(V[uj,uj]), 0, 1))
                sm_name = {(0,1): 'V_us', (1,0): 'V_cd', (0,2): 'V_ub', 
                          (2,0): 'V_td', (1,2): 'V_cb', (2,1): 'V_ts'}.get((di,uj), '')
                if sm_name:
                    print(f'    V[{di},{uj}] = {V[di,uj]:+.6f}  sin(off-diag) = {abs(V[di,uj]):.6f}  ({sm_name})')

# ══════════════════════════════════════════════════════════════
# The PHYSICAL CKM: compute sin(theta_12) from the misalignment
# between DOWN gen1-gen2 subspace and UP gen1-gen2 subspace
# ══════════════════════════════════════════════════════════════

print(f'\n=== Physical CKM angles from subspace misalignment ===')
for up_sector, up_label in [(1, 'UP-1'), (3, 'UP-3')]:
    d1 = proper_states[0][1]
    d2 = proper_states[0][2]
    d3 = proper_states[0][3]
    u1 = proper_states[up_sector][1]
    u2 = proper_states[up_sector][2]
    u3 = proper_states[up_sector][3]
    
    # Cabibbo angle: the angle between gen1_down and its projection
    # onto the gen1_up direction, measured in the gen1-gen2 plane
    # sin(theta_C) = |<d1|u2>| / sqrt(|<d1|u1>|^2 + |<d1|u2>|^2)
    
    d1_u1 = np.dot(d1, u1)
    d1_u2 = np.dot(d1, u2)
    d1_u3 = np.dot(d1, u3)
    d2_u1 = np.dot(d2, u1)
    d2_u2 = np.dot(d2, u2)
    d3_u3 = np.dot(d3, u3)
    
    print(f'\n  {up_label}:')
    print(f'    <d1|u1> = {d1_u1:.6f}')
    print(f'    <d1|u2> = {d1_u2:.6f}  ← V_us candidate')
    print(f'    <d2|u1> = {d2_u1:.6f}  ← V_cd candidate')
    print(f'    <d2|u2> = {d2_u2:.6f}')
    print(f'    <d1|u3> = {d1_u3:.6f}  ← V_ub candidate')
    print(f'    <d3|u3> = {d3_u3:.6f}')
    
    # The Cabibbo angle
    sin_C = abs(d1_u2) / np.sqrt(d1_u1**2 + d1_u2**2) if (d1_u1**2 + d1_u2**2) > 0 else 0
    print(f'    sin(theta_C) = |<d1|u2>|/sqrt(<d1|u1>^2+<d1|u2>^2) = {sin_C:.6f}')
    print(f'    PDG sin(theta_C) = 0.22500')
    
    # V_cb
    sin_cb = abs(d2_u1 * d1_u2 - d2_u2 * d1_u1)  # from the 2x2 subblock
    print(f'    |V_cb| estimate = {abs(np.dot(d2, u3)):.6f}  (PDG: 0.04053)')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.66s
=== Mass eigenstates by sector ===
For each (a5, gen), the Z2 pair has two k7 values.
The mass eigenstate is the one with larger |RMS| (4D norm).

  a5=0 (DOWN):
    gen1: dominant k7=3 (|v|=2.5856), subdominant k7=0 (|v|=0.6367)
    gen2: dominant k7=4 (|v|=3.6545), subdominant k7=1 (|v|=0.3346)
    gen3: dominant k7=5 (|v|=0.2922), subdominant k7=2 (|v|=0.2844)

  a5=1 (UP-1):
    gen1: dominant k7=3 (|v|=0.2456), subdominant k7=0 (|v|=0.2449)
    gen2: dominant k7=1 (|v|=3.4274), subdominant k7=4 (|v|=0.2499)
    gen3: dominant k7=5 (|v|=1.8073), subdominant k7=2 (|v|=0.2767)

  a5=3 (UP-3):
    gen1: dominant k7=3 (|v|=0.1297), subdominant k7=0 (|v|=0.1082)
    gen2: dominant k7=4 (|v|=1.0513), subdominant k7=1 (|v|=0.1272)
    gen3: dominant k7=2 (|v|=3.2628), subdominant k7=5 (|v|=0.1309)

=== Eigenstate misalignment (dominant k7 comparison) ===
  gen1: DOWN k7=3, UP-1 k7=3 (SAME), UP-3 k7=3 (SAME)
  gen2: DOWN

# NB163 — CKM from Cascade Dynamics

## The Mechanism

The CKM comes from the **directed Cayley splitting** (NB59), not from wrapping fractions.

The spectral wall (NB50-58) protects Gen1=Gen2 under any real Hamiltonian.
The only exit: the directed operator A_g = B_g - B_{g⁻¹}, which is Hermitian
but non-real. It shifts eigenvalues by 2ε·Im(χ(g)), where Im depends on
the full CRT index including a5 (the Z4 charge sector).

The up-type and down-type quarks live in different a5 sectors. The VEV ratio
ρ = 1/√210 breaks the Im(a5=1) = -Im(a5=3) symmetry, creating different
generation orderings in the two sectors. This misalignment IS the CKM.

## Sections
- S0: Previous results (F-N fails for |V_cb|, susceptibility matches)
- S1: Character table computation + directed eigenvalue shifts
- S2: Tower-weighted effective splitting per a5 sector
- S3: Generation ordering misalignment → CKM matrix

In [2]:
# S0 — Previous results summary
import numpy as np
from math import prod
from functools import reduce
from math import lcm as _lcm

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)  # 210
P3 = p1 * p2 * p3  # 30
kappa = 1.0 / np.sqrt(P4)
rho = kappa  # = 1/sqrt(210)
omega = 2 * np.pi
phi_P4 = prod(p-1 for p in primes)  # 48

# The Cayley generators that satisfy the gauge constraints (NB145)
generators = [17, 23, 37]

print('=== CKM Mechanism Summary ===')
print(f'Primes: {primes}, P4={P4}, rho=1/sqrt({P4})={rho:.6f}')
print(f'Generators: {generators}')
print(f'phi(P4) = {phi_P4} characters (= SM fermion states)')
print(f'\nPrevious findings:')
print(f'  F-N: sin(theta_C) = 1/sqrt(20) = {1/np.sqrt(20):.5f} (PDG: 0.22500, -2.1σ)')
print(f'  F-N: |V_cb| = sqrt(m_s/m_b) = {1/np.sqrt(44.46):.5f} (PDG: 0.04053, CATASTROPHIC MISS)')
print(f'  → F-N mass textures fail. Need directed Cayley mechanism.')
print(f'\nThe directed Cayley splitting (NB59):')
print(f'  E(chi) = lambda_L(chi) + 2*epsilon * Im(chi(g))')
print(f'  Gen1-Gen2 split = 4*epsilon * Im(chi(g))')
print(f'  The Im depends on (a3, a5, a7) → different a5 sectors get different splits')
print(f'  Up/down misalignment from rho-weighted tower interference → CKM')

=== CKM Mechanism Summary ===
Primes: [2, 3, 5, 7], P4=210, rho=1/sqrt(210)=0.069007
Generators: [17, 23, 37]
phi(P4) = 48 characters (= SM fermion states)

Previous findings:
  F-N: sin(theta_C) = 1/sqrt(20) = 0.22361 (PDG: 0.22500, -2.1σ)
  F-N: |V_cb| = sqrt(m_s/m_b) = 0.14997 (PDG: 0.04053, CATASTROPHIC MISS)
  → F-N mass textures fail. Need directed Cayley mechanism.

The directed Cayley splitting (NB59):
  E(chi) = lambda_L(chi) + 2*epsilon * Im(chi(g))
  Gen1-Gen2 split = 4*epsilon * Im(chi(g))
  The Im depends on (a3, a5, a7) → different a5 sectors get different splits
  Up/down misalignment from rho-weighted tower interference → CKM

In [3]:
# S1 — Character table of Z*_210 and directed eigenvalue shifts
#
# Z*_210 = Z1 × Z2 × Z4 × Z6 via CRT.
# Characters: chi(n) = exp(2*pi*i * (a2*k2/1 + a3*k3/2 + a5*k5/4 + a7*k7/6))
# where (a2, a3, a5, a7) is the CRT decomposition of n mod (1,3,5,7)
# and (k2, k3, k5, k7) labels the character.
#
# For generator g, Im(chi(g)) = Im(exp(2*pi*i * sum(a_p * k_p / phi_p)))
#                              = sin(2*pi * sum(a_p(g) * k_p / phi_p))

import numpy as np
from math import prod

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
rho = 1.0 / np.sqrt(P4)
phi_p = {2: 1, 3: 2, 5: 4, 7: 6}  # phi(p) = p-1
generators = [17, 23, 37]

# CRT decomposition: n mod p gives residue, then discrete log
# For p=2: Z*_2 = {1} → a2 always 0
# For p=3: Z*_3 = {1,2} → generator 2, dlog: 1→0, 2→1
# For p=5: Z*_5 = {1,2,3,4} → generator 2, dlog: 1→0, 2→1, 4→2, 3→3
# For p=7: Z*_7 = {1,2,3,4,5,6} → generator 3, dlog: 1→0, 3→1, 2→2, 6→3, 4→4, 5→5

# Discrete log tables
dlog3 = {1: 0, 2: 1}
dlog5 = {1: 0, 2: 1, 4: 2, 3: 3}
dlog7 = {1: 0, 3: 1, 2: 2, 6: 3, 4: 4, 5: 5}

def crt_decompose(n):
    """Decompose n ∈ Z*_210 into CRT indices (a2, a3, a5, a7)."""
    r3 = n % 3
    r5 = n % 5
    r7 = n % 7
    return (0, dlog3[r3], dlog5[r5], dlog7[r7])

# Verify generators
print('=== Generator CRT decomposition ===')
for g in generators:
    a = crt_decompose(g)
    print(f'  g={g}: (a2,a3,a5,a7) = {a}')

# Character evaluation: chi_{k}(g) = exp(2*pi*i * sum(a_p * k_p / phi_p))
# where k = (k2, k3, k5, k7) labels the character
# and a = CRT decomposition of g

def char_phase(a, k):
    """Phase of character k evaluated at element with CRT index a."""
    _, a3, a5, a7 = a
    _, k3, k5, k7 = k
    phase = a3 * k3 / phi_p[3] + a5 * k5 / phi_p[5] + a7 * k7 / phi_p[7]
    return 2 * np.pi * phase

def char_eval(a, k):
    """Evaluate character k at element with CRT index a."""
    return np.exp(1j * char_phase(a, k))

# Build the full character table for all 48 characters
# Characters labeled by (k2, k3, k5, k7) where k2∈{0}, k3∈{0,1}, k5∈{0,1,2,3}, k7∈{0,1,2,3,4,5}
gen_map = {0: 1, 3: 1, 1: 2, 4: 2, 2: 3, 5: 3}  # a7 → generation

print(f'\n=== Directed eigenvalue shifts Im(chi(g)) ===')
print(f'For each character (a3, a5, a7), compute the combined Im shift from 3 generators.')
print(f'The √3 ladder (NB60): at equal coupling, 4 distinct |sum Im| values.')

# For each character, compute sum of Im(chi(g)) over the 3 generators
# This is the total directed split (at equal coupling epsilon)
characters = []
for k3 in range(phi_p[3]):  # 0, 1
    for k5 in range(phi_p[5]):  # 0, 1, 2, 3
        for k7 in range(phi_p[7]):  # 0, 1, 2, 3, 4, 5
            k = (0, k3, k5, k7)
            gen = gen_map.get(k7, 0)
            
            # Compute Im(chi(g)) for each generator
            im_shifts = []
            for g in generators:
                a_g = crt_decompose(g)
                chi_val = char_eval(a_g, k)
                im_shifts.append(chi_val.imag)
            
            total_im = sum(im_shifts)
            characters.append({
                'k': k, 'k3': k3, 'k5': k5, 'k7': k7,
                'gen': gen, 'im_shifts': im_shifts,
                'total_im': total_im,
                'abs_im': abs(total_im),
            })

# Print organized by a5 sector and generation
print(f'\n{"k3":>3} {"k5":>3} {"k7":>3} {"gen":>4} {"Im(g17)":>10} {"Im(g23)":>10} {"Im(g37)":>10} {"Sum_Im":>10}')
print('-' * 65)
for c in sorted(characters, key=lambda x: (x['k5'], x['gen'], x['k7'])):
    print(f'{c["k3"]:3d} {c["k5"]:3d} {c["k7"]:3d} {c["gen"]:4d} '
          f'{c["im_shifts"][0]:10.6f} {c["im_shifts"][1]:10.6f} {c["im_shifts"][2]:10.6f} {c["total_im"]:10.6f}')

=== Generator CRT decomposition ===
  g=17: (a2,a3,a5,a7) = (0, 1, 1, 1)
  g=23: (a2,a3,a5,a7) = (0, 1, 3, 2)
  g=37: (a2,a3,a5,a7) = (0, 0, 1, 2)

=== Directed eigenvalue shifts Im(chi(g)) ===
For each character (a3, a5, a7), compute the combined Im shift from 3 generators.
The √3 ladder (NB60): at equal coupling, 4 distinct |sum Im| values.

 k3  k5  k7  gen    Im(g17)    Im(g23)    Im(g37)     Sum_Im
-----------------------------------------------------------------
  0   0   0    1   0.000000   0.000000   0.000000   0.000000
  1   0   0    1   0.000000   0.000000   0.000000   0.000000
  0   0   3    1   0.000000  -0.000000  -0.000000  -0.000000
  1   0   3    1  -0.000000   0.000000  -0.000000  -0.000000
  0   0   1    2   0.866025   0.866025   0.866025   2.598076
  1   0   1    2  -0.866025  -0.866025   0.866025  -0.866025
  0   0   4    2  -0.866025   0.866025   0.866025   0.866025
  1   0   4    2   0.866025  -0.866025   0.866025   0.866025
  0   0   2    3   0.866025  -0.866025 

In [4]:
# S2 — Tower-weighted effective splitting per a5 sector
#
# The three tower levels see different primes:
# Level 0 (C6):   primes {3}     → uses k3 only
# Level 1 (C42):  primes {3,7}   → uses k3, k7
# Level 2 (C210): primes {3,5,7} → uses k3, k5, k7
#
# Im at each level:
# Im_0 = sin(2*pi * a3(g) * k3 / 2)  [depends on k3 only]
# Im_1 = sin(2*pi * (a3(g)*k3/2 + a7(g)*k7/6))  [depends on k3, k7]
# Im_2 = sin(2*pi * (a3(g)*k3/2 + a5(g)*k5/4 + a7(g)*k7/6))  [full CRT]
#
# The tower-weighted effective splitting:
# S_eff = (1-rho) * Im_1 + rho * Im_2  (NB109/NB62)
# where rho = 1/sqrt(210)
#
# For a5=0: Im_2 = Im_1 (no a5 dependence) → S_eff = Im_1 (universal)
# For a5≠0: Im_2 ≠ Im_1 → S_eff depends on sector

import numpy as np
from math import prod

primes = [2, 3, 5, 7]
P4 = prod(primes)
rho = 1.0 / np.sqrt(P4)
phi_p = {2: 1, 3: 2, 5: 4, 7: 6}
generators = [17, 23, 37]
gen_map = {0: 1, 3: 1, 1: 2, 4: 2, 2: 3, 5: 3}

dlog3 = {1: 0, 2: 1}
dlog5 = {1: 0, 2: 1, 4: 2, 3: 3}
dlog7 = {1: 0, 3: 1, 2: 2, 6: 3, 4: 4, 5: 5}

def crt_decompose(n):
    return (0, dlog3[n%3], dlog5[n%5], dlog7[n%7])

# Compute per-level Im for each generator and character
def im_level(g, k, level):
    """Im(chi_k(g)) at a specific tower level.
    Level 0: only p=3
    Level 1: p=3, p=7
    Level 2: p=3, p=5, p=7
    """
    a = crt_decompose(g)
    _, a3, a5, a7 = a
    _, k3, k5, k7 = k
    
    if level == 0:
        phase = a3 * k3 / phi_p[3]
    elif level == 1:
        phase = a3 * k3 / phi_p[3] + a7 * k7 / phi_p[7]
    else:  # level 2
        phase = a3 * k3 / phi_p[3] + a5 * k5 / phi_p[5] + a7 * k7 / phi_p[7]
    
    return np.sin(2 * np.pi * phase)

# Compute S_eff for all 48 characters
# S_eff = sum over generators of [(1-rho)*Im_1(g,k) + rho*Im_2(g,k)]

print('=== Tower-weighted effective splitting S_eff ===')
print(f'rho = 1/sqrt({P4}) = {rho:.6f}')
print(f'S_eff = sum_g [(1-rho)*Im_1(g,k) + rho*Im_2(g,k)]')

results = []
for k3 in range(phi_p[3]):
    for k5 in range(phi_p[5]):
        for k7 in range(phi_p[7]):
            k = (0, k3, k5, k7)
            gen = gen_map.get(k7, 0)
            
            im1_total = sum(im_level(g, k, 1) for g in generators)
            im2_total = sum(im_level(g, k, 2) for g in generators)
            s_eff = (1 - rho) * im1_total + rho * im2_total
            
            results.append({
                'k3': k3, 'k5': k5, 'k7': k7, 'gen': gen,
                'im1': im1_total, 'im2': im2_total, 's_eff': s_eff,
            })

# Print organized by a5 sector
print(f'\n--- By a5 (charge) sector ---')
for k5_val in [0, 1, 2, 3]:
    label = {0: 'mass/down', 1: 'isospin-up', 2: 'protected', 3: 'isospin-down'}[k5_val]
    print(f'\n  k5={k5_val} ({label}):')
    print(f'  {"k3":>3} {"k7":>3} {"gen":>4} {"Im_1":>10} {"Im_2":>10} {"S_eff":>10} {"a5_contrib":>10}')
    sector_results = [r for r in results if r['k5'] == k5_val]
    for r in sorted(sector_results, key=lambda x: (x['gen'], x['k7'])):
        a5_contrib = r['im2'] - r['im1']  # the a5-dependent part
        print(f'  {r["k3"]:3d} {r["k7"]:3d} {r["gen"]:4d} {r["im1"]:10.6f} {r["im2"]:10.6f} {r["s_eff"]:10.6f} {a5_contrib:10.6f}')

# ══════════════════════════════════════════════════════════════
# The CKM comes from the DIFFERENCE in generation ordering
# between k5=0 (down-type) and k5=1 or k5=3 (up-type).
# ══════════════════════════════════════════════════════════════

print(f'\n=== Generation ordering by sector ===')
for k5_val in [0, 1, 2, 3]:
    label = {0: 'DOWN', 1: 'UP-1', 2: 'PROT', 3: 'UP-3'}[k5_val]
    # For each k3 (chirality), get the S_eff for each generation
    for k3_val in [0, 1]:
        chir = 'L' if k3_val == 0 else 'R'
        gen_seff = {}
        for r in results:
            if r['k5'] == k5_val and r['k3'] == k3_val:
                gen = r['gen']
                if gen not in gen_seff:
                    gen_seff[gen] = []
                gen_seff[gen].append(r['s_eff'])
        
        # Average S_eff per generation (or list if multiple)
        gen_avg = {g: np.mean(vals) for g, vals in gen_seff.items()}
        ordering = sorted(gen_avg.keys(), key=lambda g: gen_avg[g], reverse=True)
        
        print(f'  {label} {chir}: gen order by S_eff = {ordering} '
              f'(S_eff: {[f"{gen_avg[g]:.4f}" for g in ordering]})')

# ══════════════════════════════════════════════════════════════
# The mass eigenstate is determined by |S_eff|.
# The CKM measures the overlap between the DOWN and UP orderings.
# If they're the same ordering → V = Identity (no mixing).
# If they differ → V has off-diagonal elements.
# ══════════════════════════════════════════════════════════════

print(f'\n=== CKM from ordering misalignment ===')
# The down-type sector (k5=0) and up-type sector (k5=1) S_eff values
# At fixed k3 (say k3=1, right-handed quarks), extract the 6 a7 values

for k3_val in [1]:  # quarks: right-chirality
    print(f'\nChirality k3={k3_val}:')
    for k5_val in [0, 1, 3]:
        label = {0: 'DOWN', 1: 'UP-1', 3: 'UP-3'}[k5_val]
        seff_by_k7 = {}
        for r in results:
            if r['k5'] == k5_val and r['k3'] == k3_val:
                seff_by_k7[r['k7']] = r['s_eff']
        
        print(f'  {label} (k5={k5_val}):')
        for k7 in range(6):
            gen = gen_map[k7]
            s = seff_by_k7.get(k7, 0)
            print(f'    k7={k7} gen{gen}: S_eff = {s:+.6f}')
        
        # The mass eigenstates: sort by |S_eff| descending
        # (larger |S_eff| = heavier, by the tower product formula)
        sorted_k7 = sorted(seff_by_k7.keys(), key=lambda k: abs(seff_by_k7[k]), reverse=True)
        print(f'    Mass ordering (heaviest first): {[(k7, gen_map[k7], f"{seff_by_k7[k7]:+.4f}") for k7 in sorted_k7]}')

=== Tower-weighted effective splitting S_eff ===
rho = 1/sqrt(210) = 0.069007
S_eff = sum_g [(1-rho)*Im_1(g,k) + rho*Im_2(g,k)]

--- By a5 (charge) sector ---

  k5=0 (mass/down):
   k3  k7  gen       Im_1       Im_2      S_eff a5_contrib
    0   0    1   0.000000   0.000000   0.000000   0.000000
    1   0    1   0.000000   0.000000   0.000000   0.000000
    0   3    1  -0.000000  -0.000000  -0.000000   0.000000
    1   3    1  -0.000000  -0.000000  -0.000000   0.000000
    0   1    2   2.598076   2.598076   2.598076   0.000000
    1   1    2  -0.866025  -0.866025  -0.866025   0.000000
    0   4    2   0.866025   0.866025   0.866025   0.000000
    1   4    2   0.866025   0.866025   0.866025   0.000000
    0   2    3  -0.866025  -0.866025  -0.866025   0.000000
    1   2    3  -0.866025  -0.866025  -0.866025   0.000000
    0   5    3  -2.598076  -2.598076  -2.598076   0.000000
    1   5    3   0.866025   0.866025   0.866025   0.000000

  k5=1 (isospin-up):
   k3  k7  gen       Im_1      